In [ ]:
import os
import pandas as pd
from glob import glob
import concurrent.futures
import dask.dataframe as dd

code = 'SREW'

files_folder_path = f"{code}/{code}_output/"
combine_folder_path = f"{code}/{code}_output_ParameterWise/"
os.makedirs(combine_folder_path, exist_ok=True)

EXT = "*.parquet"
all_files = [file for path, subdir, files in os.walk(files_folder_path) for file in glob(os.path.join(path, EXT))]
indices = sorted(set([f.split('\\')[-1].split(' ')[0] for f in all_files]))
os.makedirs(combine_folder_path, exist_ok=True)

print('Number of Files -', len(all_files))
print()

print('Indices -', indices)
print()

df = pd.read_parquet(max([(file, os.path.getsize(file)) for file in glob(f'{files_folder_path}/*.parquet')], key=lambda x: x[-1])[0])
name_columns = [c for c in list(df.columns) if c.startswith('P_')]
print('Names cols-', list(map(lambda x: x.replace('P_', ''), name_columns)))

In [ ]:
for index in indices:
    chunk_nos = sorted(set([k.split('-')[-1].split('.')[0] for k in glob(f'{files_folder_path}/{index}*')]))
    for chunk_no in chunk_nos:
        try:
            all_files = glob(f'{files_folder_path}/{index}*No-{chunk_no}.parquet')
            print(f'\nTotal file in {index} Chunks-{chunk_no} -', len(all_files))
            if len(all_files) == 0:continue

            print('Reading Chunks...')
            df = dd.read_parquet(all_files)
            df = df.compute()
            print('Reading Complete...')

            os.makedirs(f"{combine_folder_path}{index}", exist_ok=True)
            print('Grouping Chunks...')
            grouped = df.groupby(name_columns)

            def save_file(idx, data):
                try:
                    file_name = ' '.join(map(str, idx)).replace(":00 ", " ").replace(":", "") + '.parquet'
                    data.to_parquet(f"{combine_folder_path}{index}/{file_name}", index=False)
                except Exception as e:
                    print(e)

            with concurrent.futures.ThreadPoolExecutor(max_workers=os.cpu_count()*7) as executor:
                for idx, data in grouped:
                    executor.submit(save_file, idx, data)
        except Exception as e:
            input(str(e))